## LSTM Power Consumption Prediction

### 1. Import libraries

In [16]:
print("Hello World")

Hello World


In [17]:
import re
from collections import Counter
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.preprocessing import MinMaxScaler

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


### 2. Import Dataset

In [18]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
individual_household_electric_power_consumption = fetch_ucirepo(id=235) 
  
# data (as pandas dataframes) 
X_raw = individual_household_electric_power_consumption.data.features 

c:\Users\doubl\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0: Global_active_power, 1: Global_reactive_power, 2: Voltage, 3: Global_intensity, 4: Sub_metering_1, 5: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


### 3. Deal with Missing Values (Clean Dataset)

In [34]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler


# ============================================================
# 1. COPY RAW DATA
# ============================================================

X = X_raw.copy()


# ============================================================
# 2. HANDLE MISSING VALUES
# ============================================================

X = X.replace("?", np.nan)


# ============================================================
# 3. CREATE DATETIME INDEX
# ============================================================

X["datetime"] = pd.to_datetime(
    X["Date"] + " " + X["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

X = X.set_index("datetime")

print("Missing values by column:")
print(X.isna().sum())


# ============================================================
# 4. CONVERT SENSOR COLUMNS TO NUMERIC
# ============================================================

cols_to_interpolate = [
    "Global_active_power",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for col in cols_to_interpolate:

    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

    # Time-based interpolation
    X[col] = X[col].interpolate(
        method="time"
    )


# ============================================================
# 5. CALCULATE MINUTE-LEVEL ENERGY
# ============================================================

X["power"] = (
    X["Global_active_power"] * 1000 / 60
    - X["Sub_metering_1"]
    - X["Sub_metering_2"]
    - X["Sub_metering_3"]
)


# ============================================================
# 6. RESAMPLE TO HOURLY
# ============================================================

X_hrs = X["power"].resample("h").mean()


# ============================================================
# 7. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

end_date = X_hrs.index.max()

# Final 6 months = test
test_start = (
    end_date
    - pd.DateOffset(months=6)
)

# 6 months before test = validation
val_start = (
    end_date
    - pd.DateOffset(months=12)
)


# Training
X_train_raw = X_hrs.loc[
    X_hrs.index < val_start
]


# Validation
X_val_raw = X_hrs.loc[
    (X_hrs.index >= val_start)
    &
    (X_hrs.index < test_start)
]


# Test
X_test_raw = X_hrs.loc[
    X_hrs.index >= test_start
]


# ============================================================
# 8. CHECK SPLITS
# ============================================================

print(
    "Train:",
    X_train_raw.index.min(),
    "to",
    X_train_raw.index.max()
)

print(
    "Validation:",
    X_val_raw.index.min(),
    "to",
    X_val_raw.index.max()
)

print(
    "Test:",
    X_test_raw.index.min(),
    "to",
    X_test_raw.index.max()
)


print(
    "\nNumber of observations:"
)

print(
    "Train:",
    len(X_train_raw)
)

print(
    "Validation:",
    len(X_val_raw)
)

print(
    "Test:",
    len(X_test_raw)
)


# ============================================================
# 9. SCALE DATA
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, 1)
)


# FIT ONLY ON TRAINING DATA
X_train = scaler.fit_transform(
    X_train_raw.values.reshape(-1, 1)
)


# TRANSFORM VALIDATION
X_val = scaler.transform(
    X_val_raw.values.reshape(-1, 1)
)


# TRANSFORM TEST
X_test = scaler.transform(
    X_test_raw.values.reshape(-1, 1)
)

Missing values by column:
Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64
Train: 2006-12-16 17:00:00 to 2009-11-26 20:00:00
Validation: 2009-11-26 21:00:00 to 2010-05-26 20:00:00
Test: 2010-05-26 21:00:00 to 2010-11-26 21:00:00

Number of observations:
Train: 25828
Validation: 4344
Test: 4417


### 4. Split into Periods for Training

In [35]:
past_length = 168           # Length of relevant data in hours
forcast_length = 24         # Want to predict 24 hours in advance

def get_sequence(data):
    X_out = []
    y_out = []

    for i in range(0, len(data) - past_length - forcast_length + 1, 24):
        X_out.append(data[i:i+past_length])
        y_out.append(data[i+past_length:i+past_length+forcast_length])

    X_out = np.array(X_out)
    y_out = np.array(y_out)

    return X_out, y_out

class ProcessedDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = ProcessedDataset(*get_sequence(X_train))
train_loader = DataLoader(train_dataset)
val_loader = DataLoader(ProcessedDataset(*get_sequence(X_val)))
test_loader = DataLoader(ProcessedDataset(*get_sequence(X_test)))



In [36]:
class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=512, num_layers=1, batch_first=True)
        self.fc = nn.Linear(512, 24)

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        return self.fc(out[:, -1, :])

model = SimpleLSTM()


In [40]:
import copy
import torch
import torch.nn as nn

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = model.to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

num_epochs = 10

# Track best validation model
best_val_loss = float("inf")
best_model_state = None


# ============================================================
# TRAINING
# ============================================================

for epoch in range(num_epochs):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        # Move data to GPU
        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        y_batch = y_batch.to(
            device,
            non_blocking=True
        )

        # Remove final dimension
        # (batch, 24, 1) -> (batch, 24)
        y_batch = y_batch.squeeze(-1)

        # Reset gradients
        optimizer.zero_grad(
            set_to_none=True
        )

        # Forward pass
        predictions = model(
            X_batch
        )

        # Calculate loss
        loss = criterion(
            predictions,
            y_batch
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        train_loss += loss.item()


    # Average training loss
    train_loss /= len(train_loader)


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(
                device,
                non_blocking=True
            )

            y_batch = y_batch.to(
                device,
                non_blocking=True
            )

            y_batch = y_batch.squeeze(-1)

            predictions = model(
                X_batch
            )

            loss = criterion(
                predictions,
                y_batch
            )

            val_loss += loss.item()


    # Average validation loss
    val_loss /= len(val_loader)


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        print(
            f"  New best model! "
            f"Val Loss: {best_val_loss:.6f}"
        )


# ============================================================
# RESTORE BEST MODEL
# ============================================================

model.load_state_dict(
    best_model_state
)

model.to(device)

print(
    f"\nBest validation loss: "
    f"{best_val_loss:.6f}"
)

Using device: cuda
Epoch 01/10 | Train Loss: 0.013159 | Val Loss: 0.020927
  New best model! Val Loss: 0.020927
Epoch 02/10 | Train Loss: 0.010707 | Val Loss: 0.024880
Epoch 03/10 | Train Loss: 0.010277 | Val Loss: 0.025716
Epoch 04/10 | Train Loss: 0.010027 | Val Loss: 0.025825
Epoch 05/10 | Train Loss: 0.009891 | Val Loss: 0.026171
Epoch 06/10 | Train Loss: 0.009834 | Val Loss: 0.026096
Epoch 07/10 | Train Loss: 0.009816 | Val Loss: 0.026584
Epoch 08/10 | Train Loss: 0.009803 | Val Loss: 0.025675
Epoch 09/10 | Train Loss: 0.009922 | Val Loss: 0.026343
Epoch 10/10 | Train Loss: 0.009937 | Val Loss: 0.025691

Best validation loss: 0.020927


In [42]:
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        predictions = model(X_batch)

        all_predictions.append(
            predictions.cpu()
        )

        all_actuals.append(
            y_batch.squeeze(-1)
        )

all_predictions = torch.cat(
    all_predictions
).numpy()

all_actuals = torch.cat(
    all_actuals
).numpy()

predictions_original = scaler.inverse_transform(
    all_predictions.reshape(-1, 1)
).reshape(
    all_predictions.shape
)

actuals_original = scaler.inverse_transform(
    all_actuals.reshape(-1, 1)
).reshape(
    all_actuals.shape
)

predictions_original = scaler.inverse_transform(
    all_predictions.reshape(-1, 1)
).reshape(
    all_predictions.shape
)

actuals_original = scaler.inverse_transform(
    all_actuals.reshape(-1, 1)
).reshape(
    all_actuals.shape
)

epsilon = 1e-6

mask = np.abs(actuals_original) > epsilon

mape = np.mean(
    np.abs(
        (actuals_original[mask] -
         predictions_original[mask])
        /
        actuals_original[mask]
    )
) * 100

print(
    f"Test MAPE: {mape:.2f}%"
)

Test MAPE: 100.84%
